In [ ]:
from gaussians import Gaussian
from jax import numpy as jnp
from jaxtyping import Array, Float

In [ ]:
def GEQRF(A: Float[Array, "M N"], prior: Gaussian) -> (Callable, Float[Array, "M M"], Float[Array, "M N"], Float[Array, "M M"]):
    M, N = A.shape # dimensions
    U, R, nu = jnp.zeros((M, N)), jpn.zeros((N, N)), jnp.zeros(N) # storage
    Sigma = prior.Sigma
    for n in range(N): # matrix decomposition
        an = A[:, n]
        un = Sigma @ an
        un = un / jnp.sqrt(jnp.dot(an, un))
        U = U.at[:, n].set(un)
        R = R.at[: n+1, n].set(an.T @ U[:, : n+1])
        nu = nu.at[n].set(jnp.dot(an, prior.mu))
        Sigma = Sigma - jnp.outer(un, un)

    def solve(b: Float[Array, "n"]) -> Gaussian: # solver routine
        alpha = jnp.zeros((N,))
        alpha = alpha.at[0].set((b[0] - nu[0]) / R[0, 0])
        for n in range(1, N):
            alpha = alpha.at[n].set((b[n] - nu[n] - jnp.dot(alpha[:n], R[:n, n])) / R[n, n])
        return Gaussian(prior.mu + U @ alpha, prior.Sigma)
    return solve, R, U, Sigma